In [4]:
# model_evaluation.py

from dataclasses import dataclass
from pathlib import Path
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_from_disk
import evaluate
import os
import sys

# Try to import sentencepiece, warn if not installed
try:
    import sentencepiece
except ImportError:
    print("⚠️ sentencepiece not installed. Tokenizer may not load correctly.")
    print("   Run: pip install sentencepiece")


# -------------------------------------------------------------------
# Direct configuration (edit these paths to match your system)
# -------------------------------------------------------------------
@dataclass
class ModelEvaluationConfig:
    root_dir: str
    data_path: str
    model_path: str
    tokenizer_path: str
    metric_file_name: str


def get_direct_config():
    """Return a configuration object with your actual paths."""
    return ModelEvaluationConfig(
        root_dir=r"d:/projects/Text Summarizer/artifacts/model_evaluation",
        data_path=r"d:/projects/Text Summarizer/artifacts/data_transformation/samsum_dataset",
        model_path=r"d:/projects/Text Summarizer/notebooks/model_trainer",      # Your trained model
        tokenizer_path=r"d:/projects/Text Summarizer/notebooks/model_trainer",  # Usually same as model
        metric_file_name=r"d:/projects/Text Summarizer/artifacts/model_evaluation/metrics.csv"
    )


# -------------------------------------------------------------------
# Evaluation class
# -------------------------------------------------------------------
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i: i + batch_size]

    def calculate_metric_on_test_ds(
        self,
        dataset,
        metric,
        model,
        tokenizer,
        batch_size=16,
        device="cuda" if torch.cuda.is_available() else "cpu",
        column_text="dialogue",
        column_summary="summary",
    ):
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches), total=len(article_batches)
        ):
            inputs = tokenizer(
                article_batch,
                max_length=1024,
                truncation=True,
                padding="max_length",
                return_tensors="pt",
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}

            summaries = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                length_penalty=0.8,
                num_beams=8,
                max_length=128,
            )

            decoded_summaries = [
                tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True)
                for s in summaries
            ]

            metric.add_batch(predictions=decoded_summaries, references=target_batch)

        score = metric.compute()
        return score

    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {device}")

        # Check if tokenizer path exists
        if not os.path.exists(self.config.tokenizer_path):
            raise FileNotFoundError(f"Tokenizer path not found: {self.config.tokenizer_path}")

        # Try loading tokenizer with fallback
        print(f"\nLoading tokenizer from: {self.config.tokenizer_path}")
        try:
            tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path, use_fast=True)
            print("✓ Tokenizer loaded (fast)")
        except Exception as e:
            print(f"Fast tokenizer failed: {e}")
            try:
                tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path, use_fast=False)
                print("✓ Tokenizer loaded (slow)")
            except Exception as e2:
                print(f"Slow tokenizer also failed: {e2}")
                print("\n💡 Possible solutions:")
                print("   1. Install sentencepiece: pip install sentencepiece")
                print("   2. Check that the tokenizer files exist at the path above.")
                print("   3. Ensure the tokenizer was saved correctly during training.")
                sys.exit(1)

        # Load model
        if not os.path.exists(self.config.model_path):
            raise FileNotFoundError(f"Model path not found: {self.config.model_path}")
        print(f"Loading model from: {self.config.model_path}")
        model = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)
        print("✓ Model loaded")

        # Load dataset
        if not os.path.exists(self.config.data_path):
            raise FileNotFoundError(f"Data path not found: {self.config.data_path}")
        print(f"Loading dataset from: {self.config.data_path}")
        dataset = load_from_disk(self.config.data_path)
        print(f"Test set size: {len(dataset['test'])}")

        # Load ROUGE metric
        rouge_metric = evaluate.load("rouge")

        # Evaluate on a subset (first 10 examples for speed; adjust as needed)
        test_subset = dataset["test"].select(range(min(10, len(dataset["test"]))))

        score = self.calculate_metric_on_test_ds(
            test_subset,
            rouge_metric,
            model,
            tokenizer,
            batch_size=2,
            column_text="dialogue",
            column_summary="summary",
        )

        # Extract ROUGE scores
        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
        rouge_dict = {}
        for rn in rouge_names:
            if rn in score:
                if hasattr(score[rn], "mid"):
                    rouge_dict[rn] = score[rn].mid.fmeasure
                elif isinstance(score[rn], dict) and "mid" in score[rn]:
                    rouge_dict[rn] = score[rn]["mid"].get("fmeasure", score[rn]["mid"])
                else:
                    rouge_dict[rn] = score[rn]

        # Save results
        os.makedirs(os.path.dirname(self.config.metric_file_name), exist_ok=True)
        df = pd.DataFrame([rouge_dict], index=["pegasus"])
        df.to_csv(self.config.metric_file_name, index=False)

        print(f"\n✓ Metrics saved to: {self.config.metric_file_name}")
        print("ROUGE Scores:", rouge_dict)

        return rouge_dict


# -------------------------------------------------------------------
# Run evaluation
# -------------------------------------------------------------------
if __name__ == "__main__":
    config = get_direct_config()
    evaluator = ModelEvaluation(config)
    evaluator.evaluate()

Using device: cpu

Loading tokenizer from: d:/projects/Text Summarizer/notebooks/model_trainer
Fast tokenizer failed: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.
Slow tokenizer also failed: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

💡 Possible solutions:
   1. Install sentencepiece: pip install sentencepiece
   2. Check that the tokenizer files exist at the path above.
   3. Ensure the tokenizer was saved correctly during training.


SystemExit: 1

c:\Users\Nikhil\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
